# Cross-Dataset Generalization + Federated Tree Benchmark (UQ NetFlow v2)

**Aim:** Test genuine leave-one-source-dataset-out network intrusion detection using the harmonized **NF-UQ-NIDS-v2** NetFlow schema. Compare centralized LightGBM, local-domain LightGBM ensembles, and federated histogram-statistic boosting. Report feature-shortcut sensitivity to ports and TTL.

**Required Kaggle input:** Attach an official `NF-UQ-NIDS-v2.csv` file (combined, containing the `Dataset` column) **or** all four constituent `NF-*-v2.csv` files from the University of Queensland. The previously used `Edge-IIoTset` CSV alone is **not compatible** and will not be silently substituted.

Official downloads: https://staff.itee.uq.edu.au/marius/NIDS_datasets/

**Scientific constraints:** No target-domain labels in fitting or calibration; fixed target evaluation per seed; feature definitions must match at the NetFlow v2 extraction level. Sampling caps balance source/domain classes and change test prevalence; report FPR and recall as diagnostics, not deployment precision. The federated method transmits local gradient/Hessian histograms, but offers **no formal privacy protection**. All training is CPU-oriented; GPU acceleration is not required.

In [ ]:
import os, sys, importlib.util, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/nf_cross_notebook_test')
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['NF_CROSS_ROOT'] = str(ROOT)
for key in ['OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[key] = '1'
print('Python:', sys.version.split()[0])
print('Output root:', ROOT)
try:
    import lightgbm
    print('LightGBM:', lightgbm.__version__)
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','lightgbm'])
    import lightgbm
    print('LightGBM installed:', lightgbm.__version__)

## Embedded engine

The next cell materializes the complete standalone engine. You do **not** need to upload a separate `.py` file. Normal runs must have the real NetFlow v2 input attached. Smoke fixtures are only used when `NF_CROSS_SMOKE=1` is explicitly set for developer tests.

In [ ]:
%%writefile /kaggle/working/nf_cross_engine.py
"""NF-UQ-NIDS cross-source generalization: pooled GBDT, local ensemble, histogram-fed boosting.
The source-dataset holdout is genuine only if canonical UQ NetFlow schema and dataset provenance exist.
No raw records are shared by FedHistogramGB. Histograms are not privacy guarantees.
English-only, unattended Kaggle notebook. Tested with synthetic data for software correctness only.
"""
import os
for k in ('OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):os.environ[k]='1'
import csv,json,sys,hashlib,zipfile,traceback,argparse,time,faulthandler
from pathlib import Path
from collections import defaultdict,Counter
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import roc_auc_score,average_precision_score
from sklearn.model_selection import train_test_split
faulthandler.enable()
ROOT=Path(os.getenv('NF_CROSS_ROOT','/kaggle/working' if Path('/kaggle/working').is_dir() else '/mnt/data/nf_cross_local'))
OUT=ROOT/'NF_CrossDomain_Results';OUT.mkdir(parents=True,exist_ok=True)
CACHE=ROOT/'nf_cross_cache.npz'
TARGETS=['NF-UNSW-NB15','NF-ToN-IoT','NF-BoT-IoT','NF-CSE-CIC-IDS2018']
FEATURES=['L4_SRC_PORT','L4_DST_PORT','PROTOCOL','L7_PROTO','IN_BYTES','OUT_BYTES','IN_PKTS','OUT_PKTS','FLOW_DURATION_MILLISECONDS','TCP_FLAGS','CLIENT_TCP_FLAGS','SERVER_TCP_FLAGS','DURATION_IN','DURATION_OUT','MIN_TTL','MAX_TTL','LONGEST_FLOW_PKT','SHORTEST_FLOW_PKT','MIN_IP_PKT_LEN','MAX_IP_PKT_LEN','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES','RETRANSMITTED_IN_BYTES','RETRANSMITTED_IN_PKTS','RETRANSMITTED_OUT_BYTES','RETRANSMITTED_OUT_PKTS','SRC_TO_DST_AVG_THROUGHPUT','DST_TO_SRC_AVG_THROUGHPUT','NUM_PKTS_UP_TO_128_BYTES','NUM_PKTS_128_TO_256_BYTES','NUM_PKTS_256_TO_512_BYTES','NUM_PKTS_512_TO_1024_BYTES','NUM_PKTS_1024_TO_1514_BYTES','TCP_WIN_MAX_IN','TCP_WIN_MAX_OUT','ICMP_TYPE','ICMP_IPV4_TYPE','DNS_QUERY_ID','DNS_QUERY_TYPE','DNS_TTL_ANSWER','FTP_COMMAND_RET_CODE']
REQUIRED=['L4_SRC_PORT','L4_DST_PORT','PROTOCOL','IN_BYTES','OUT_BYTES','IN_PKTS','OUT_PKTS','FLOW_DURATION_MILLISECONDS','TCP_FLAGS']
SETS={'netflow_full_no_ip':FEATURES,'no_ports':[f for f in FEATURES if f not in ('L4_SRC_PORT','L4_DST_PORT')], 'no_ports_no_ttl':[f for f in FEATURES if f not in ('L4_SRC_PORT','L4_DST_PORT','MIN_TTL','MAX_TTL')]}
METHODS=['PooledLightGBM','LocalLightGBM','FedHistogramGB']
CONFIG={'seeds':[0,42,3407],'holdouts':TARGETS,'methods':METHODS,'feature_sets':list(SETS),'targets_val_fpr':[.01,.05],
        'max_rows_per_domain_class':12000,'rows_per_source_validation_class':2000,'batch_rows':4000,
        'hist_bins':16,'federated_rounds':55,'fed_lr':.15,'fed_l2':3.,'lightgbm_trees':90,
        'privacy_note':'FedHistogramGB aggregates local gradient/hessian histograms, but does not provide secure aggregation or differential privacy.',
        'protocol_note':'Independent source-dataset holdout on harmonized UQ NetFlow v2 fields. Not a direct Edge-to-NetFlow model transfer.'}
if os.getenv('NF_CROSS_SMOKE')=='1':CONFIG.update(seeds=[0],holdouts=TARGETS[:2],feature_sets=['netflow_full_no_ip','no_ports'],max_rows_per_domain_class=350,rows_per_source_validation_class=60,hist_bins=8,federated_rounds=8,lightgbm_trees=18)
def jsave(p,v):Path(p).write_text(json.dumps(v,indent=2,default=str))
def header(p):
 with open(p,'r',encoding='utf-8-sig',errors='replace',newline='') as f:return next(csv.reader(f))
def normalize(s):return ''.join(c for c in str(s).lower() if c.isalnum())
def canonical_domain(s):
 z=str(s).lower().replace('_','-').replace(' ','')
 if 'unsw' in z:return TARGETS[0]
 if 'ton' in z and 'iot' in z:return TARGETS[1]
 if 'bot' in z and 'iot' in z:return TARGETS[2]
 if 'cse' in z or 'cic' in z or 'ids2018' in z:return TARGETS[3]
 return None

def discover_files():
 roots=[Path('/kaggle/input'),ROOT/'input']
 if os.getenv('NF_UQ_DATA'):
  roots.insert(0,Path(os.environ['NF_UQ_DATA']))
 found=[]
 for root in roots:
  if not root.exists():continue
  paths=[root] if root.is_file() else list(root.rglob('*.csv'))
  for p in paths:
   if 'nf-' not in str(p).lower() and 'nf_' not in str(p).lower() and 'netflow' not in str(p).lower():continue
   # Not the dataset description metadata file.
   if 'features' in p.name.lower():continue
   try:
    h=header(p);hs=set(h)
    if set(REQUIRED).issubset(hs) and ('Label' in hs or 'label' in hs):found.append((p,h))
   except Exception:continue
 return sorted(found,key=lambda ph:(0 if 'nf-uq-nids-v2' in ph[0].name.lower() else 1,len(str(ph[0]))))

def synthetic_source():
 """STRICTLY a local software smoke fixture; never a research benchmark."""
 p=ROOT/'synthetic_nf_software_test.csv';ROOT.mkdir(parents=True,exist_ok=True)
 if p.exists():return p
 rng=np.random.default_rng(17)
 with open(p,'w',newline='') as file:
  wr=csv.writer(file);wr.writerow(FEATURES+['Label','Attack','Dataset'])
  for d,domain in enumerate(TARGETS):
   for c in [0,1]:
    for i in range(440):
     v=rng.gamma(2,1,len(FEATURES));v[0]=int(80+30*d+10*c);v[1]=int(443+13*d);v[4]+=c*(2+d*.7);v[5]+=c*.6;v[6]+=c*.8
     wr.writerow([f'{x:.5f}' for x in v]+[str(c),'DDoS' if c else 'Benign',domain+'-v2'])
 return p

def stream_files(paths):
 rng=np.random.default_rng(2026);reservoir=defaultdict(list);seen=Counter();invalid=Counter();perfile=[]
 for p,h in paths:
  label=next((q for q in h if normalize(q)=='label'),None)
  dataset=next((q for q in h if normalize(q)=='dataset'),None)
  attack=next((q for q in h if normalize(q)=='attack'),None)
  fixed=canonical_domain(p.name)
  if not label or not (dataset or fixed):
   print('SKIPPING_SOURCE_NO_DOMAIN_OR_LABEL',p.name,label,dataset,fixed,flush=True);continue
  ix={c:h.index(c) for c in FEATURES};li=h.index(label);di=h.index(dataset) if dataset else None;ai=h.index(attack) if attack else None
  valid=0;bad=0
  with open(p,'r',encoding='utf-8-sig',errors='replace',newline='',buffering=1024*1024) as f:
   rd=csv.reader(f);next(rd)
   for row in rd:
    if len(row)!=len(h):bad+=1;continue
    domain=canonical_domain(row[di]) if di is not None else fixed
    if domain not in TARGETS:invalid['unknown_domain']+=1;continue
    raw=row[li].strip().lower()
    if raw in ('0','benign','normal'):y=0
    elif raw in ('1','attack','malicious'):y=1
    else:invalid['unknown_label']+=1;continue
    values=[]
    try:
     for col in FEATURES:
      z=row[ix[col]].strip();v=float(z) if z else 0.
      values.append(v if np.isfinite(v) else 0.)
    except Exception:bad+=1;continue
    key=(domain,y);seen[key]+=1
    n=seen[key];buffer=reservoir[key];rec=(np.asarray(values,dtype=np.float32),str(row[ai]) if ai is not None else '')
    if len(buffer)<CONFIG['max_rows_per_domain_class']:buffer.append(rec)
    else:
     j=int(rng.integers(n))
     if j<len(buffer):buffer[j]=rec
    valid+=1
  perfile.append(dict(path=str(p),valid_rows=valid,malformed_or_non_numeric_rows=bad))
  print('SOURCE_SCANNED',p.name,'valid',valid,'bad',bad,flush=True)
 data=[];labels=[];domains=[];attacks=[]
 for (domain,y),records in sorted(reservoir.items()):
  for arr,att in records:data.append(arr);labels.append(y);domains.append(domain);attacks.append(att)
 if not data:raise ValueError('NO_VALID_RECORDS_FROM_NF_DATASET')
 x=np.asarray(data,dtype=np.float32);y=np.asarray(labels,dtype=np.int8);domain=np.asarray(domains);attack=np.asarray(attacks)
 np.savez_compressed(CACHE,x=x,y=y,domain=domain,attack=attack)
 provenance={'source_files':perfile,'population_counts':{f'{d}|{c}':int(v) for (d,c),v in seen.items()},'reservoir_counts':{f'{d}|{c}':len(v) for (d,c),v in reservoir.items()},'invalid':dict(invalid),'feature_list':FEATURES,'sampling':'uniform reservoir per domain x binary class; training/validation/test observed prevalences are altered by sampling','source_schema':'UQ NetFlow V2 41 non-IP fields only, not Edge packet-level fields','research_external_domain_definition':'source dataset, not device/client','max_rows_per_domain_class':CONFIG['max_rows_per_domain_class']}
 jsave(OUT/'dataset_provenance.json',provenance)
 # Must have all four domains and two classes, with enough examples for heldout tests.
 counts=Counter(zip(domain,y));minimum=40 if os.getenv('NF_CROSS_SMOKE')=='1' else 100
 problems=[f'{d}/{c}={counts[(d,c)]}' for d in TARGETS for c in [0,1] if counts[(d,c)]<minimum]
 jsave(OUT/'data_coverage.json',{'counts':{f'{d}/{c}':counts[(d,c)] for d in TARGETS for c in [0,1]},'requirements_met':not problems,'problems':problems})
 if problems:raise RuntimeError('INSUFFICIENT_SOURCE_DOMAIN_COVERAGE: '+', '.join(problems))
 return x

def prepare():
 jsave(OUT/'config.json',CONFIG)
 # These packet features cannot be assumed equivalent to NetFlow aggregates.
 edge={'tcp.srcport':'L4_SRC_PORT','tcp.dstport':'L4_DST_PORT','tcp.flags':'TCP_FLAGS','tcp.len':'IN_BYTES','dns.qry.type':'DNS_QUERY_TYPE','udp.time_delta':'FLOW_DURATION_MILLISECONDS'}
 with open(OUT/'edge_to_netflow_schema_audit.csv','w',newline='') as f:
  w=csv.writer(f);w.writerow(['edge_column','suggested_netflow_column','semantically_certified','reason'])
  for a,b in edge.items():w.writerow([a,b,'NO','Packet-level / direction / aggregation / extraction semantics differ or are unverified'])
 if os.getenv('NF_CROSS_SMOKE')=='1':files=[(synthetic_source(),header(synthetic_source()))]
 else:files=discover_files()
 jsave(OUT/'discovery.json',{'files':[str(p) for p,h in files], 'source':'attached NF-UQ-NIDS-v2 only','note':'Attach canonical NF-UQ-NIDS-v2 combined CSV or constituent NF-v2 files; Edge-IIoTset alone is insufficient.'})
 if not files:raise FileNotFoundError('NO_NF_UQ_V2_FILES: Attach NF-UQ-NIDS-v2.csv in Kaggle Add Input. Edge-IIoTset CSV cannot be substituted.')
 # Do not double-count combined and individual constituent files.
 combined=[ph for ph in files if 'nf-uq-nids-v2' in ph[0].name.lower()]
 if combined:files=[combined[0]]
 elif os.getenv('NF_CROSS_SMOKE')=='1':pass
 else:
  selected=[];used=set()
  for p,h in files:
   d=canonical_domain(p.name)
   if d and d not in used:selected.append((p,h));used.add(d)
  files=selected
 stream_files(files)

def build_splits(y,domains,held,seed):
 tr=[];va=[];te=np.flatnonzero(domains==held)
 for d in TARGETS:
  if d==held:continue
  ids=np.flatnonzero(domains==d)
  a,b=train_test_split(ids,test_size=.2,random_state=seed,stratify=y[ids])
  tr.extend(a);va.extend(b)
 tr=np.asarray(tr);va=np.asarray(va)
 assert len(set(tr)&set(va))==0 and len(set(tr)&set(te))==0
 assert set(domains[tr])==set(TARGETS)-{held}
 return tr,va,te

def transform(x,indices):
 # Fixed unit-free log transform; no target statistics or fitting.
 raw=np.asarray(x[:,indices],dtype=np.float64)
 return np.clip(np.sign(raw)*np.log1p(np.abs(np.clip(raw,-1e12,1e12))),-18,18).astype('float32')

def metric(y,p,t):
 yy=np.asarray(y,dtype=int);pred=(p>=t);tp=int(((pred==1)&(yy==1)).sum());fp=int(((pred==1)&(yy==0)).sum());fn=int(((pred==0)&(yy==1)).sum());tn=int(((pred==0)&(yy==0)).sum())
 return dict(tp=tp,fp=fp,fn=fn,tn=tn,unknown_recall=tp/max(1,tp+fn),test_fpr=fp/max(1,tn+fp),precision=tp/max(1,tp+fp),attack_f1=2*tp/max(1,2*tp+fp+fn),auroc=float(roc_auc_score(yy,p)) if len(np.unique(yy))==2 else np.nan,auprc=float(average_precision_score(yy,p)) if len(np.unique(yy))==2 else np.nan)

def val_threshold(p,y,target):
 benign=np.asarray(p)[np.asarray(y)==0]
 if len(benign)<20:raise ValueError('TOO_FEW_BENIGN_VALIDATION')
 # Conservative strict exceedance threshold handles heavily tied tree probabilities.
 t=float(np.quantile(benign,1-target,method='higher'))
 return float(np.nextafter(t,np.inf))

class FedHistogramGB:
 def __init__(self,rounds=55,bins=16,lr=.15,l2=3.):self.rounds=rounds;self.bins=bins;self.lr=lr;self.l2=l2;self.communication_bytes=0;self.nodes=[]
 def fit(self,x,y,clients):
  n,d=x.shape;self.edges=[]
  # Each client reports only per-feature minima/maxima, then uses broadcast shared bin edges.
  # No pooled sample quantiles or raw examples are used for the bin-edge calculation.
  mins=np.stack([x[idx].min(0) for idx in clients if len(idx)])
  maxs=np.stack([x[idx].max(0) for idx in clients if len(idx)])
  low=mins.min(0);high=maxs.max(0)
  for j in range(d):
   edges=np.linspace(float(low[j]),float(high[j])+1e-5,self.bins+1)[1:-1]
   self.edges.append(edges)
  self.communication_bytes+=len(clients)*d*2*4 + len(clients)*d*(self.bins-1)*4
  bx=np.column_stack([np.searchsorted(e,x[:,j],side='right') for j,e in enumerate(self.edges)]).astype('uint8')
  p0=(float(np.mean(y))*(len(y))+1)/(len(y)+2);self.bias=float(np.log(p0/(1-p0)))
  logits=np.full(n,self.bias)
  for r in range(self.rounds):
   # Aggregate only histogram sums, not raw client rows. Bin edges are shared.
   G=np.zeros((d,self.bins));H=np.zeros_like(G)
   for ids in clients:
    if not len(ids):continue
    prob=expit(logits[ids]);g=prob-y[ids];h=np.maximum(prob*(1-prob),1e-6)
    for j in range(d):
     gg=np.bincount(bx[ids,j],weights=g,minlength=self.bins)
     hh=np.bincount(bx[ids,j],weights=h,minlength=self.bins)
     G[j]+=gg[:self.bins];H[j]+=hh[:self.bins]
    self.communication_bytes+=d*self.bins*2*8
   lg=np.cumsum(G,axis=1)[:,:-1];lh=np.cumsum(H,axis=1)[:,:-1];rg=G.sum(1,keepdims=True)-lg;rh=H.sum(1,keepdims=True)-lh
   gain=.5*(lg**2/(lh+self.l2)+rg**2/(rh+self.l2)-(lg+rg)**2/(lh+rh+self.l2))
   gain[(lh<2)|(rh<2)]=-np.inf
   if not np.isfinite(gain).any():break
   j,k=np.unravel_index(np.argmax(gain),gain.shape)
   if gain[j,k]<=0:break
   left=float(-lg[j,k]/(lh[j,k]+self.l2)*self.lr);right=float(-rg[j,k]/(rh[j,k]+self.l2)*self.lr)
   self.nodes.append((int(j),int(k),left,right));self.communication_bytes+=len(clients)*24
   logits+=np.where(bx[:,j]<=k,left,right)
  return self
 def predict(self,x):
  bins=np.column_stack([np.searchsorted(e,x[:,j],side='right') for j,e in enumerate(self.edges)])
  z=np.full(len(x),self.bias)
  for j,k,left,right in self.nodes:z+=np.where(bins[:,j]<=k,left,right)
  return expit(z)

def train_method(method,x,y,domains,tr,va,te,seed):
 from lightgbm import LGBMClassifier
 tic=time.perf_counter();bytes_sent=0;models=0
 def new_model(s):return LGBMClassifier(n_estimators=CONFIG['lightgbm_trees'],num_leaves=15,max_depth=5,min_child_samples=15,learning_rate=.06,random_state=s,n_jobs=1,verbosity=-1)
 if method=='PooledLightGBM':
  m=new_model(seed);m.fit(x[tr],y[tr]);pv=m.predict_proba(x[va])[:,1];pt=m.predict_proba(x[te])[:,1];models=1
 elif method=='LocalLightGBM':
  vals=[];tests=[]
  for k,domain in enumerate(TARGETS):
   ids=tr[domains[tr]==domain]
   if len(ids)<50 or len(np.unique(y[ids]))<2:continue
   m=new_model(seed+k);m.fit(x[ids],y[ids]);vals.append(m.predict_proba(x[va])[:,1]);tests.append(m.predict_proba(x[te])[:,1])
  if not vals:raise ValueError('NO_VALID_SOURCE_CLIENTS')
  pv=np.mean(vals,0);pt=np.mean(tests,0);models=len(vals)
 else:
  xx=x[tr];yy=y[tr];ds=domains[tr];clients=[np.flatnonzero(ds==d) for d in TARGETS if d in set(ds)]
  model=FedHistogramGB(CONFIG['federated_rounds'],CONFIG['hist_bins'],CONFIG['fed_lr'],CONFIG['fed_l2']).fit(xx,yy,clients)
  pv=model.predict(x[va]);pt=model.predict(x[te]);bytes_sent=model.communication_bytes;models=len(model.nodes)
 return pv,pt,dict(seconds=time.perf_counter()-tic,communication_bytes=bytes_sent,n_models_or_stumps=models,training_mode={'PooledLightGBM':'centralized pooled reference','LocalLightGBM':'local domain ensemble; deployment communication not counted','FedHistogramGB':'federated histogram statistics; no privacy guarantee'}[method])

def worker(idx,workers):
 data=np.load(CACHE);x0=data['x'];y=data['y'];dom=data['domain'];attack=data['attack']
 jobs=[(held,seed,fs,method) for held in CONFIG['holdouts'] for seed in CONFIG['seeds'] for fs in CONFIG['feature_sets'] for method in METHODS]
 jobs=[j for i,j in enumerate(jobs) if i%workers==idx]
 rows=[];res=[];fail=[];cache={};splits={}
 for k,(held,seed,fs,method) in enumerate(jobs,1):
  try:
   if (held,seed) not in splits:splits[(held,seed)]=build_splits(y,dom,held,seed)
   tr,va,te=splits[(held,seed)]
   if fs not in cache:cache[fs]=transform(x0,[FEATURES.index(f) for f in SETS[fs]])
   x=cache[fs]
   pv,pt,r=train_method(method,x,y,dom,tr,va,te,seed)
   test_hash=hashlib.sha256(np.sort(te).tobytes()).hexdigest();val_hash=hashlib.sha256(np.sort(va).tobytes()).hexdigest()
   for tar in CONFIG['targets_val_fpr']:
    th=val_threshold(pv,y[va],tar)
    m=metric(y[te],pt,th)
    rows.append(dict(heldout_domain=held,seed=seed,feature_set=fs,method=method,target_val_fpr=tar,validation_threshold=th,validation_realized_fpr=float((pv[y[va]==0]>=th).mean()),n_train=len(tr),n_val=len(va),n_test=len(te),source_domains=','.join(sorted(set(dom[tr]))),test_hash=test_hash,val_hash=val_hash,**m))
   res.append(dict(heldout_domain=held,seed=seed,feature_set=fs,method=method,n_train=len(tr),n_val=len(va),n_test=len(te),**r))
  except Exception as e:fail.append({'job':str((held,seed,fs,method)),'error':repr(e),'traceback':traceback.format_exc()[-2200:]})
  if k%6==0:print('WORKER_PROGRESS',idx,k,'/',len(jobs),'completed',len(res),'errors',len(fail),flush=True)
 if rows:pd.DataFrame(rows).to_csv(OUT/f'worker_{idx}_metrics.csv',index=False)
 if res:pd.DataFrame(res).to_csv(OUT/f'worker_{idx}_resources.csv',index=False)
 jsave(OUT/f'worker_{idx}_failures.json',fail)
 print('WORKER_DONE',idx,'completed',len(res),'failed',len(fail),flush=True)
 if fail:sys.exit(2)

def aggregate(workers):
 def collect(name):
  v=[]
  for i in range(workers):
   p=OUT/f'worker_{i}_{name}.csv'
   if p.is_file() and p.stat().st_size>10:v.append(pd.read_csv(p))
  return pd.concat(v,ignore_index=True) if v else pd.DataFrame()
 m=collect('metrics');r=collect('resources');fail=[]
 for i in range(workers):
  p=OUT/f'worker_{i}_failures.json'
  if p.is_file():fail.extend(json.loads(p.read_text()))
 expected=len(CONFIG['holdouts'])*len(CONFIG['seeds'])*len(CONFIG['feature_sets'])*len(METHODS)
 status={'expected_jobs':expected,'completed_jobs':len(r),'failed_jobs':len(fail),'status':'complete' if len(r)==expected and not fail else 'incomplete','notes':['External test uses an actual withheld UQ source dataset, not the original Edge dataset.','Data are per-domain class-balanced by capped reservoir sampling, not real deployment class prevalence.','No external target labels used for fitting, hyperparameter selection, or threshold selection.','NF-v2 and Edge packet-level features are NOT treated as equivalent.']}
 if len(m):
  m.to_csv(OUT/'all_metrics.csv',index=False);r.to_csv(OUT/'all_resources.csv',index=False)
  inv=m.groupby(['heldout_domain','seed']).agg(test_hashes=('test_hash','nunique'),val_hashes=('val_hash','nunique'))
  status['fixed_test_invariance']=bool((inv.test_hashes==1).all() and (inv.val_hashes==1).all())
  summary=m.groupby(['heldout_domain','feature_set','method','target_val_fpr']).agg(seed_count=('seed','nunique'),recall_mean=('unknown_recall','mean'),recall_std=('unknown_recall','std'),fpr_mean=('test_fpr','mean'),auroc_mean=('auroc','mean'),auprc_mean=('auprc','mean'),f1_mean=('attack_f1','mean')).reset_index()
  summary.to_csv(OUT/'summary_mean_std.csv',index=False)
  pairs=[]
  for key,g in m.groupby(['heldout_domain','seed','feature_set','target_val_fpr']):
   p=g[g.method=='PooledLightGBM']
   if p.empty:continue
   for _,z in g.iterrows():
    if z.method=='PooledLightGBM':continue
    pairs.append(dict(heldout_domain=key[0],seed=key[1],feature_set=key[2],target_val_fpr=key[3],method=z.method,delta_recall_vs_pooled=float(z.unknown_recall-p.iloc[0].unknown_recall),delta_fpr_vs_pooled=float(z.test_fpr-p.iloc[0].test_fpr),delta_auroc_vs_pooled=float(z.auroc-p.iloc[0].auroc)))
  if pairs:pd.DataFrame(pairs).to_csv(OUT/'paired_method_deltas.csv',index=False)
  try:
   import matplotlib;matplotlib.use('Agg');import matplotlib.pyplot as plt
   a=summary[(summary.feature_set=='netflow_full_no_ip')&(summary.target_val_fpr==.05)]
   fig,ax=plt.subplots(figsize=(11,5))
   for met,g in a.groupby('method'):ax.plot(g.heldout_domain,g.recall_mean,marker='o',label=met)
   ax.set_ylim(0,1);ax.set_ylabel('Unknown-domain attack recall');ax.tick_params(axis='x',rotation=25);ax.legend();fig.tight_layout();fig.savefig(OUT/'domain_transfer_recall.png',dpi=150);plt.close(fig)
  except Exception as e:print('PLOT_ERROR',e,flush=True)
 jsave(OUT/'run_status.json',status)
 with zipfile.ZipFile(ROOT/'NF_CrossDomain_Results.zip','w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
  for p in OUT.rglob('*'):
   if p.is_file():z.write(p,p.relative_to(ROOT))
 print('AGGREGATION',status,'ZIP',ROOT/'NF_CrossDomain_Results.zip',flush=True)
 if status['status']!='complete' or not status.get('fixed_test_invariance'):raise RuntimeError('INCOMPLETE_OR_INVALID_CROSS_DOMAIN_BENCHMARK')

def main():
 pa=argparse.ArgumentParser();pa.add_argument('--mode',choices=['prepare','worker','aggregate'],required=True);pa.add_argument('--worker',type=int,default=0);pa.add_argument('--workers',type=int,default=1);a=pa.parse_args()
 if a.mode=='prepare':prepare()
 elif a.mode=='worker':worker(a.worker,a.workers)
 else:aggregate(a.workers)
if __name__=='__main__':main()


In [ ]:
import subprocess, traceback, zipfile, json
from pathlib import Path
engine=Path('/kaggle/working/nf_cross_engine.py')
if not engine.exists():
    # Local notebook-execution smoke test only; Kaggle always uses /kaggle/working.
    engine=ROOT/'nf_cross_engine.py'
    if not engine.exists():
        raise FileNotFoundError('Embedded engine is missing')
out=ROOT/'NF_CrossDomain_Results';out.mkdir(exist_ok=True)

def invoke(args, logfile):
    env=os.environ.copy()
    with open(out/logfile, 'w') as stream:
        p=subprocess.run([sys.executable,str(engine),*args],env=env,stdout=stream,stderr=subprocess.STDOUT)
    tail=(out/logfile).read_text(errors='replace')[-6000:]
    print(logfile, 'exit=',p.returncode, '\n'+tail,flush=True)
    if p.returncode:
        raise RuntimeError(f'{logfile} failed; read {out/logfile}. It is not valid to substitute Edge-IIoTset for UQ NetFlow.')

invoke(['--mode','prepare'],'prepare.log')
worker_count=2
workers=[]
for i in range(worker_count):
    f=open(out/f'worker_{i}.log','w')
    p=subprocess.Popen([sys.executable,str(engine),'--mode','worker','--worker',str(i),'--workers',str(worker_count)],
                       stdout=f,stderr=subprocess.STDOUT,env=os.environ.copy())
    workers.append((i,p,f))
failed=[]
for i,p,f in workers:
    code=p.wait();f.close()
    print('Worker',i,'exit',code,'\n',(out/f'worker_{i}.log').read_text(errors='replace')[-2500:],flush=True)
    if code:failed.append(i)
if failed:raise RuntimeError(f'Worker failures: {failed}. Check worker logs and failure JSON in {out}.')
invoke(['--mode','aggregate','--workers',str(worker_count)],'aggregate.log')
zip_path=ROOT/'NF_CrossDomain_Results.zip'
print('OUTPUT:',zip_path,'size bytes:',zip_path.stat().st_size)
from IPython.display import display,FileLink
display(FileLink(str(zip_path)))

In [ ]:
import pandas as pd, json
from pathlib import Path
out=ROOT/'NF_CrossDomain_Results'
status=json.loads((out/'run_status.json').read_text())
print('Research run status:',json.dumps(status,indent=2))
summary=pd.read_csv(out/'summary_mean_std.csv')
display(summary[(summary.feature_set=='netflow_full_no_ip')&(summary.target_val_fpr==0.05)].sort_values(['heldout_domain','method']))
print('NOTE: The software smoke test is synthetic and must never be cited as a real research result.')